# MPPP — 05 Error analysis of COLMAP alignments (MPPP 0.41.0)

Takes one or more alignments made by notebook 03 (their `colmap/error_input/` folders) and measures the numbers the error model (`mppp.error`) runs on, next to the values it assumes (`ModelConfig` defaults):

| section | measured | model value |
|---|---|---|
| 2 image precision | ε = rms residual / √2 per instrument, same-station vs cross-station tracks, against range and against convergence angle | `ModelConfig` ε |
| 3 cross-station gate | match rate vs convergence angle θ and an illumination covariate; the angle form and the covariate are chosen by AIC and every parameter is fitted | `ModelConfig` gate |
| 4 decorrelation | ρ(θ) of pairwise triangulation residuals (pairs sharing no image, bias-corrected, with standard errors) and its fit | `ModelConfig` ρ_0, θ_c |
| 5 view graph | shared tracks between stations, connectivity | — |
| 6 registration | bundle-adjusted pose − telemetry prior, per station | 0.3 m (telemetry registration) |

**How the gate is measured.** For a random sample of reconstructed points, every image pair in which the point is geometrically visible (in front of both cameras, inside both frames) is a candidate. A trial is: the point is matched in image *a*; is it also matched in image *b*? The same-station rate is the reference and A is relative to it, as in the model. Terrain that never matched in either image (occluded, masked rover, no texture) stays out of the denominator (`pair_survival(conditional=False)` counts it too).

**Caveats.** Sparse tracks are biased towards well-matched terrain, so ε is a lower bound (post-fit residuals) and the survival rates are upper bounds. The gate is only measured where the rate actually falls inside 0–`THETA_MAX_DEG`; the fit reports `constrained = False` otherwise. A needs pairs at small \|ΔLMST\| and the illumination e-fold needs a spread of the covariate; with few stations the angle and illumination terms are confounded with station identity (the bootstrap range says so). The illumination e-fold in hours of \|ΔLMST\| is called τ (`tau_h`); L_s is the solar longitude and is not used here.

Figures and tables go to `OUT`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import sys, json, time, datetime
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import numpy as np
import matplotlib.pyplot as plt
import mppp
from mppp.error import alignment as EA
from mppp.error.viewgraph import graph_report
print("mppp", mppp.__version__, "from", Path(mppp.__file__).parent)

try:
    import pandas as pd
    from IPython.display import display
    def show(rows, cols=None, digits=3):
        df = pd.DataFrame(rows)
        if cols:
            df = df[[c for c in cols if c in df]]
        display(df.round(digits))
        return df
except ImportError:                                   # plain printout without pandas
    pd = None
    def show(rows, cols=None, digits=3):
        cols = cols or list(rows[0]) if rows else []
        print("  ".join(f"{c:>14}" for c in cols))
        for r in rows:
            print("  ".join(f"{r.get(c):>14.{digits}f}" if isinstance(r.get(c), float) else f"{str(r.get(c)):>14}"
                            for c in cols))
        return rows

## 1  Alignments

With `ALIGNMENTS = None` (default) every site folder below `SCAPES_ROOT` with a finished notebook 03 alignment and its `colmap/error_input/` is used, in sol order, labelled from the folder name (`mppp.sfm.sites.discover_scapes`; the table lists every folder and why one was left out: no finished alignment or a run in progress, no `error_input`, fewer than 10 images, or no image in the site's sol range). A dict {name: folder} — the WORK folder of notebook 03, its `colmap` folder or the `error_input` folder itself — analyses just those; missing ones are skipped. Exports from before 0.14.5 work too (they have no Mastcam-Z instrument split).

Every loaded alignment stays in memory (about 0.3 GB for a Navcam scape); for many alignments on a small machine, `scripts/error_analysis_batch.py` makes the same measurements one alignment at a time.

`MIN_TRACK_LENGTH` (default 3) keeps only tie points seen in at least that many images. The two-view points (about half of all points, mostly single stereo pairs) are then left out. That applies to every section below: ε, the gate, decorrelation, the view graph. Restricting to longer tracks also raises the conditional match rates, because a point seen in three images is more likely to be seen in a fourth; compare with `MIN_TRACK_LENGTH = 2` before reading the gate as absolute.

In [ ]:
SCAPES_ROOT = Path(r"D:\scapes\colmap")   # notebook 03's WORK folders
ALIGNMENTS = None        # None: every finished alignment with an error_input/ below SCAPES_ROOT, in sol order
                         # (mppp.sfm.sites.discover_scapes); or a dict {name: folder} to analyse just those
SCAPE_EXCLUDE = []       # site folders, sites or labels to leave out, e.g. ["whale_mountain"]
INCLUDE_ZCAM34 = False   # also the <site>_colmap_nav_zcam34 blocks ("<Site> N+Z34")
SITES = None             # sol ranges for the folder check (None: mppp.sfm.sites.SITES, as notebook 03)
MIN_TRACK_LENGTH = 3     # v0p22: only tie points seen in >= 3 images (2 = all, incl. the two-view points)
N_POINTS      = 5000     # points sampled per alignment for the gate (about 10 s per 1000 points for 300 images)
N_BOOT        = 50       # bootstrap resamples of those points for the gate's 16-84 % range (0 = none)
THETA_MAX_DEG = 30.0     # the gate is fitted over 0 .. this convergence angle
FIT_TAU       = True     # fit the illumination e-fold (False: hold it at tau_h = 2.3 h)
GATE_FORM     = "power"  # angle form: "power" (1 + θ/θ_c)^-k, "exp", "stretch", "logistic"; section 3b compares them
ILLUMINATION  = "dlmst"  # covariate: "dlmst" |ΔLMST| [h], "sunangle" angle between the sun vectors [deg], "shadow", "none"
SEED          = 0
OUT = Path(r"D:\scapes\error_analysis") / datetime.date.today().isoformat()

In [ ]:
from mppp.sfm.sites import discover_scapes
if ALIGNMENTS is None:
    ALIGNMENTS = discover_scapes(SCAPES_ROOT, sites=SITES, require_error_input=True, include_zcam=INCLUDE_ZCAM34,
                                 exclude=SCAPE_EXCLUDE)
print(len(ALIGNMENTS), "alignments:", ", ".join(ALIGNMENTS))


In [ ]:
OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)
als = {}
for name, path in ALIGNMENTS.items():
    try:
        t = time.time()
        als[name] = EA.load_alignment(path, label=name, min_track_length=MIN_TRACK_LENGTH)
        print(f"{name}: {len(als[name].model.images)} images, {len(als[name].model.points)} points "
              f"(of {als[name].points_before_track_filter}; tracks >= {MIN_TRACK_LENGTH} images), "
              f"{len(als[name].stations)} stations  ({time.time() - t:.0f} s)  <- {als[name].root}")
    except FileNotFoundError as e:
        print(f"{name}: skipped ({e})")
assert als, "no alignment found"
print("output:", OUT)

In [ ]:
rows = []
for name, al in als.items():
    s, tr = al.summary, al.summary.get("tracks", {})
    fams = sorted({r["family"] for r in al.images.values()})
    rows.append({"alignment": name, "cameras": "+".join(fams), "images": len(al.model.images),
                 "registered": s.get("registered_images"), "stations": len(al.stations), "points": len(al.model.points),
                 "cross_station_fraction": tr.get("cross_station_fraction"), "mean_track_length": tr.get("mean_track_length"),
                 "residual_rms_px": s.get("residual_rms_native_px"), "components": len(s.get("station_components") or [])})
overview = show(rows)

## 2  Image-measurement precision ε

ε = rms(reprojection residual)/√2: one image axis, in native pixels (the model's `per_image` convention). The residuals are post-fit. `eps_dof_px` corrects for the three coordinates each point absorbs: ε·√(2N/(2N − 3P)) for N observations of P points. That makes alignments with different track lengths comparable. "intra" = tracks inside one station, "cross" = tracks that span stations. Section 2b puts ε against the convergence angle of the track (largest ray angle) and of the observation (angle to the nearest other ray), same-station and cross-station apart: whether precision degrades with convergence, separately from completeness (the gate, section 3).

In [ ]:
eps_rows = [r for al in als.values() for r in EA.eps_table(al)]
eps = show(eps_rows, ["alignment", "group", "tracks", "n_obs", "n_points", "eps_px", "eps_dof_px", "rms_px", "median_px"])

fig, axes = plt.subplots(1, len(als), figsize=(4.2 * len(als), 3.4), squeeze=False, sharey=True)
for ax, (name, al) in zip(axes[0], als.items()):
    fams = sorted({r["family"] for r in al.images.values()})
    for fam in fams:
        er = EA.eps_by_range(al, family=fam)
        for kind, ls in (("intra", "-"), ("cross", "--")):
            ok = er[f"n_{kind}"] >= 100
            ax.plot(er["range_m"][ok], er[f"eps_{kind}_px"][ok], ls, marker="o", ms=3, label=f"{fam} {kind}")
    ax.set_title(name, fontsize=10); ax.set_xlabel("range [m]"); ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("ε [native px]"); axes[0, 0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "eps_vs_range.png", dpi=150); plt.show()

### 2b  ε against convergence angle

In [ ]:
ea_rows = [r for al in als.values() for r in EA.eps_by_angle(al)]
fig, axes = plt.subplots(1, len(als), figsize=(4.4 * len(als), 3.4), squeeze=False, sharey=True)
# cross-station curves first (they lie above the same-station ones), each series with a fixed colour; the legend
# lists the series from the highest to the lowest mean ε of the first panel, i.e. in the order the curves appear
SERIES = [("cross", "theta_nn", "--", "s", "C3"), ("cross", "theta_max", "--", "o", "C2"),
          ("intra", "theta_max", "-", "o", "C0"), ("intra", "theta_nn", "-", "s", "C1")]
first = {}
for j, (ax, (name, al)) in enumerate(zip(axes[0], als.items())):
    for kind, angle, ls, mk, col in SERIES:
        rr = [r for r in ea_rows if r["alignment"] == name and r["tracks"] == kind and r["angle"] == angle]
        if rr:
            h, = ax.plot([0.5 * (r["theta_lo"] + r["theta_hi"]) for r in rr], [r["eps_px"] for r in rr], ls, marker=mk, ms=3,
                         color=col, label=f"{kind}, {'largest ray angle' if angle == 'theta_max' else 'nearest ray'}")
            if j == 0:
                first[h] = float(np.mean([r["eps_px"] for r in rr]))
    ax.set_xscale("log"); ax.set_title(name, fontsize=10); ax.set_xlabel("convergence angle [deg]"); ax.grid(alpha=0.3, which="both")
hs = sorted(first, key=first.get, reverse=True)
axes[0, 0].set_ylabel("ε [native px]"); axes[0, 0].legend(hs, [h.get_label() for h in hs], fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "eps_vs_theta.png", dpi=150); plt.show()
eps_theta = show(ea_rows, ["alignment", "tracks", "angle", "theta_lo", "theta_hi", "n", "eps_px", "median_px"])

## 3  Cross-station gate

Per alignment: the same-station rate (reference, grey) and the cross-station rate against convergence angle, with the fitted gate (evaluated at the pairs' own illumination covariate). With Mastcam-Z in the alignment, the gate is also fitted per camera pairing.

Read `theta_half_deg` first: the angle at which the cross-station rate has halved, whatever the shape. `constrained = False` means the angle form ran to a limit (CV → 0: a sharper cut-off than the form allows); `chi2_dof` says how well the form fits the 1° bins. A is the cross-station rate at zero angle and zero illumination difference relative to the same-station rate (`A_absolute` is the rate itself).

In [ ]:
pairs = {}
for name, al in als.items():
    t = time.time()
    pairs[name] = EA.pair_survival(al, n_points=N_POINTS, seed=SEED)
    p = pairs[name]
    print(f"{name}: {p['observed'].size} candidate pairs ({int(p['cross'].sum())} cross-station) "
          f"from {p['n_points']} points, {time.time() - t:.0f} s")

In [ ]:
MIN_TRIALS = 2000          # cross-station trials needed to fit a camera pairing
fits = []
for name, p in pairs.items():
    fams = [None] + sorted({f for f in np.unique(p["families"]) if f and len(np.unique(p["families"])) > 1})
    for fam in fams:
        sel = EA._select(p, fam, True)
        if p["weight"][sel].sum() < MIN_TRIALS:
            continue
        g = EA.fit_gate(p, families=fam, fit_tau=FIT_TAU, theta_max_deg=THETA_MAX_DEG, n_boot=N_BOOT, seed=SEED,
                        form=GATE_FORM, illumination=ILLUMINATION)
        fits.append(dict(g, alignment=name))
# pooled over the alignments, Navcam-Navcam pairs
if len(pairs) > 1:
    pooled = EA.combine_pairs(list(pairs.values()))
    g = EA.fit_gate(pooled, families="Navcam-Navcam", fit_tau=FIT_TAU, theta_max_deg=THETA_MAX_DEG, n_boot=N_BOOT,
                    seed=SEED, form=GATE_FORM, illumination=ILLUMINATION)
    fits.append(dict(g, alignment="pooled"))
gate_cols = ["alignment", "families", "form", "illumination", "theta_half_deg", "constrained", "chi2_dof", "A", "A_16_84",
             "theta_bar_deg", "theta_bar_deg_16_84", "cv", "cv_16_84", "beta", "theta_0_deg", "width_deg", "tau_h",
             "tau_h_16_84", "sun0_deg", "s_intra", "A_absolute", "n_cross_pairs", "matched_cross_pairs", "dlmst_h_range"]
gates = show(fits, [c for c in gate_cols if any(c in f for f in fits)])
defaults = EA.model_defaults()
print("model (ModelConfig): A %.2f, theta_bar %.1f deg, CV %.2f, tau %.1f h" %
      (defaults["gate_A"], defaults["theta_bar_deg"], defaults["gate_cv"], defaults["tau_h"]))
bad = [f"{f['alignment']} ({f['families']})" for f in fits if "A" in f and not f.get("constrained", True)]
if bad:
    print("angle form at a limit (CV -> 0 or theta_bar >= %g deg): %s -- compare theta_half_deg instead" % (THETA_MAX_DEG, ", ".join(bad)))
poor = [f"{f['alignment']} ({f['families']}): {f['chi2_dof']:.0f}" for f in fits if "A" in f and f["chi2_dof"] > 10]
if poor:
    print("chi2/dof > 10 (the form does not describe the 1-deg bins):", "; ".join(poor))

### 3b  Which angle form and which illumination covariate

Every angle form (power, exponential, stretched exponential, logistic) with every covariate (none, |ΔLMST|, sun-vector angle) fitted to the same binned trials; ΔAIC from the best. All parameters are fitted. A site with only a few stations cannot tell the forms apart (the fits run to limits); the pooled row is the one to read.

In [ ]:
form_rows = []
for name, p in pairs.items():
    for r in EA.compare_gate_forms(p, theta_max_deg=THETA_MAX_DEG):
        form_rows.append(dict(r, alignment=name))
if len(pairs) > 1:
    for r in EA.compare_gate_forms(pooled, families="Navcam-Navcam", theta_max_deg=THETA_MAX_DEG):
        form_rows.append(dict(r, alignment="pooled"))
forms = show(form_rows, ["alignment", "form", "illumination", "dAIC", "chi2_dof", "n_params", "A", "theta_bar_deg", "cv", "beta",
                         "theta_0_deg", "width_deg", "tau_h", "sun0_deg"])

In [ ]:
edges = np.arange(0, THETA_MAX_DEG + 1, 1.0)
fig, axes = plt.subplots(1, len(pairs), figsize=(4.4 * len(pairs), 3.6), squeeze=False, sharex=True, sharey=True)
for ax, (name, p) in zip(axes[0], pairs.items()):
    intra = EA.survival_curve(p, edges, cross=False)
    cross = EA.survival_curve(p, edges, cross=True)
    ok = intra["n_possible"] >= 50
    ax.plot(intra["theta_deg"][ok], intra["rate"][ok], "o", color="0.6", ms=3, label="same station")
    ok = cross["n_possible"] >= 50
    ax.errorbar(cross["theta_deg"][ok], cross["rate"][ok], [cross["rate"][ok] - cross["lo"][ok], cross["hi"][ok] - cross["rate"][ok]],
                fmt="o", ms=3, color="C0", label="cross station")
    f = next((x for x in fits if x["alignment"] == name and x["families"] == "all" and "A" in x), None)
    if f:
        pred = EA.gate_expected(p, f, edges)
        lab = f"{f['form']} fit A {f['A']:.2f}" + (f" θ̄ {f['theta_bar_deg']:.1f}°" if "theta_bar_deg" in f else "") + \
              (f" CV {f['cv']:.2f}" if "cv" in f else "") + (f" τ {f['tau_h']:.1f} h" if np.isfinite(f.get("tau_h", np.nan)) else "")
        ax.plot(cross["theta_deg"], pred, "-", color="C3", label=lab + ("" if f.get("constrained", True) else " (unconstrained)"))
        if np.isfinite(f["theta_half_deg"]):
            ax.axvline(f["theta_half_deg"], color="C3", lw=0.6, ls=":")
    ax.set_title(name, fontsize=10); ax.set_xlabel("convergence angle θ [deg]"); ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes[0, 0].set_xlim(0, THETA_MAX_DEG); axes[0, 0].set_ylim(0, None)     # shared: every panel on the same axes
axes[0, 0].set_ylabel("P(matched in b | matched in a)")
fig.tight_layout(); fig.savefig(OUT / "gate.png", dpi=150); plt.show()

In [ ]:
# |dLMST|: cross-station rate at small angle (theta < 5 deg) against the LMST difference of the two images
fig, ax = plt.subplots(figsize=(5.2, 3.6))
dl_edges = np.arange(0, 9, 0.5)
for name, p in pairs.items():
    sel = EA._select(p, None, True) & (p["theta_deg"] < 5)
    dl = np.nan_to_num(p["dlmst_h"][sel], nan=-1)
    k = np.digitize(dl, dl_edges) - 1
    ok = (k >= 0) & (k < dl_edges.size - 1)
    w = p["weight"][sel][ok].astype(float)
    n = np.bincount(k[ok], weights=w, minlength=dl_edges.size - 1)
    s = np.bincount(k[ok], weights=w * p["observed"][sel][ok], minlength=dl_edges.size - 1)
    m = n >= 100
    ax.plot(0.5 * (dl_edges[:-1] + dl_edges[1:])[m], (s / np.maximum(n, 1))[m], "o-", ms=3, label=name)
x = np.linspace(0, 8, 50)
for name in pairs:
    f = next((x_ for x_ in fits if x_["alignment"] == name and x_["families"] == "all" and np.isfinite(x_.get("tau_h", np.nan))), None)
    if f:
        ax.plot(x, f["A_absolute"] * np.exp(-x / f["tau_h"]), "--", lw=0.9, label=f"{name}: fitted τ {f['tau_h']:.1f} h")
ax.set_xlabel("|ΔLMST| [h]"); ax.set_ylabel("cross-station rate, θ < 5°"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "gate_dlmst.png", dpi=150); plt.show()

## 4  Decorrelation ρ(θ)

For tracks of 4–12 images: the point is triangulated from every image pair; ρ is the correlation of those pair residuals against the angle between the pairs' bisectors, for pairs that share no image (pairs sharing an image are correlated by that image's own error whatever the angle; that curve is dotted grey), with the −1/(M−1) constraint bias of residuals about their mean removed and standard errors from a bootstrap over tracks. `fit_rho` fits ρ_∞ + ρ_0·f(θ/θ_c) (Gaussian and exponential). The first bin says whether ρ is consistent with zero at all.

In [ ]:
rhos, rho_fits = {}, []
fig, ax = plt.subplots(figsize=(6.0, 3.8))
for k, (name, al) in enumerate(als.items()):
    t = time.time()
    rhos[name] = r = EA.decorrelation(al, n_points=min(N_POINTS, 6000), seed=SEED, n_boot=N_BOOT or 100)
    fr = EA.fit_rho(r)
    rho_fits.append({"alignment": name, "n_tracks": r["n_tracks"], **{f"first_bin_{a}": b for a, b in fr["first_bin"].items()},
                     **{f"{a}": b for a, b in fr["gaussian"].items()}, "exp_theta_c_deg": fr["exponential"].get("theta_c_deg"),
                     "exp_rho_0": fr["exponential"].get("rho_0"), "exp_chi2_dof": fr["exponential"].get("chi2_dof")})
    ok = r["n"] >= 50
    ax.errorbar(r["theta_deg"][ok], r["rho"][ok], r["se"][ok], fmt="o-", ms=3, color=f"C{k}", label=f"{name} ({r['n_tracks']} tracks)")
    ax.plot(r["theta_deg"][ok], r["rho_shared_image"][ok], ":", color="0.6", lw=0.8)
    g = fr["gaussian"]
    if "rho_0" in g:
        th = np.geomspace(0.03, 40, 200)
        ax.plot(th, g["rho_inf"] + g["rho_0"] * np.exp(-th ** 2 / (2 * g["theta_c_deg"] ** 2)), "--", color=f"C{k}", lw=0.9,
                label=f"  fit ρ_0 {g['rho_0']:.2f} θ_c {g['theta_c_deg']:.2f}° ρ_∞ {g['rho_inf']:.3f}")
    print(f"{name}: {time.time() - t:.0f} s; first bin ρ = {fr['first_bin']['rho']:.3f} ± {fr['first_bin']['se']:.3f}"
          f" ({'consistent' if fr['first_bin']['consistent_with_zero'] else 'not consistent'} with zero)")
ax.axhline(0, color="k", lw=0.6)
ax.set_xscale("log"); ax.set_xlabel("separation of pair bisectors [deg]"); ax.set_ylabel("ρ (pairs sharing no image)")
ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "rho.png", dpi=150); plt.show()
rho_table = show(rho_fits)

## 5  Measured view graph

Edge weight = tracks shared by two stations. λ₂ = 0 means the stations are not all tied together; articulation stations are single points of failure.

In [ ]:
graphs = {}
fig, axes = plt.subplots(1, len(als), figsize=(4.4 * len(als), 4.0), squeeze=False)
for ax, (name, al) in zip(axes[0], als.items()):
    graphs[name] = g = EA.view_graph(al)
    print(f"--- {name}\n" + graph_report(g))
    labels = [al.station_label(s) for s in g.names]
    im = ax.imshow(np.log10(1 + g.W), cmap="viridis")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=6)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)
    ax.set_title(name, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.7, label="log10(1 + shared tracks)")
fig.tight_layout(); fig.savefig(OUT / "view_graph.png", dpi=150); plt.show()

## 6  Registration: bundle adjustment vs telemetry

Median over each station's images of |C_BA − C_prior| (images with < 30 observations left out). The BA is tied to the priors (σ 1 m by default), so these are lower bounds on the telemetry error. The model's FIXED case uses a registration error of 0.3 m.

In [ ]:
reg_rows = [r for al in als.values() for r in EA.registration(al)]
reg = show(reg_rows, ["alignment", "station_label", "images", "dC_median_m", "dE_median_m", "dN_median_m", "dU_median_m",
                      "dAttitude_median_deg"])
for name, al in als.items():
    for c in al.summary.get("prior_similarity_by_component", []):
        print(f"{name}: similarity prior -> BA over {len(c['stations'])} stations: scale {c['scale_refined_over_prior']:.4f}, "
              f"rotation {c['rotation_deg']:.2f} deg, residual per station {c['station_residual_after_similarity_m']}")

## 7  Summary

One row per alignment, beside the model's values. Where a gate is `constrained`, `model_config` gives the `ModelConfig` arguments to run the error model with the measured numbers.

In [ ]:
summary_rows = []
for name, al in als.items():
    g = next((f for f in fits if f["alignment"] == name and f["families"] in ("all", "Navcam-Navcam") and "A" in f), None)
    summary_rows.append(EA.parameters(al, g, rhos.get(name)))
summary = show(summary_rows)
d = {k: v for k, v in defaults.items() if k != "cases"}
print("model:", json.dumps(d))

model_config = {}
for f in fits:
    if "A" in f and f["constrained"]:
        model_config[f"{f['alignment']} [{f['families']}]"] = {"gate_A": round(f["A"], 3), "theta_bar_deg": round(f["theta_bar_deg"], 2),
                                                               "gate_cv": round(f["cv"], 3), "tau_h": round(f["tau_h"], 2)}
print(json.dumps(model_config, indent=1))

def _j(o):
    return o.item() if isinstance(o, np.generic) else o.tolist() if isinstance(o, np.ndarray) else str(o)
(OUT / "error_analysis.json").write_text(json.dumps({
    "mppp_version": mppp.__version__, "alignments": {k: str(v.root) for k, v in als.items()},
    "settings": {"MIN_TRACK_LENGTH": MIN_TRACK_LENGTH, "N_POINTS": N_POINTS, "N_BOOT": N_BOOT, "THETA_MAX_DEG": THETA_MAX_DEG, "FIT_TAU": FIT_TAU, "SEED": SEED},
    "eps": eps_rows, "gate_fits": fits, "registration": reg_rows, "summary": summary_rows, "model": defaults,
    "model_config": model_config,
    "rho": {k: {"theta_deg": v["theta_deg"], "rho": v["rho"], "n": v["n"]} for k, v in rhos.items()}}, indent=1, default=_j),
    encoding="utf-8")
if pd is not None:
    pd.DataFrame(summary_rows).to_csv(OUT / "summary.csv", index=False)
    pd.DataFrame(fits).to_csv(OUT / "gate_fits.csv", index=False)
    pd.DataFrame(eps_rows).to_csv(OUT / "eps.csv", index=False)
print("written:", sorted(p.name for p in OUT.iterdir()))